# 02 - MASt3R Hybrid Matching (Per-Dataset)

## Overview
- **Hybrid MASt3R matching** matching the 1st place solution:
  - **Dense**: MASt3R semi-dense matching (subsample=8, pixel_tol=5)
  - **ALIKED Sparse**: ALIKED keypoints (max 4096, resize 1280) matched via MASt3R descriptor interpolation
  - **SuperPoint Sparse**: SuperPoint keypoints (max 4096, threshold 0.0005, resize 1600) matched via MASt3R descriptor interpolation
- All three match sources combined (union with deduplication)
- Process each dataset independently
- Output: per-dataset match files for COLMAP reconstruction

## 1. Environment & Config

In [1]:
import os, sys, json, time, gc
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

IMAGE_ROOT = Path('../image-matching-challenge-2025/small_train')
RETRIEVAL_DIR = Path('../output/retrieval')
OUTPUT_DIR = Path('../output/mast3r_matching')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# MAST3R dense matching params (1st place writeup)
MATCH_SIZE = 384
SUBSAMPLE = 8
PIXEL_TOL = 5
MATCH_THRESHOLD = 1.001  # dot-product threshold for reciprocal NN
MIN_MATCHES = 15
USE_AMP = device.type == 'cuda'
EMPTY_CACHE_EVERY_PAIR = True

# Hybrid matching — additional keypoint detectors fed to MAST3R
ALIKED_MAX_KP = 4096
ALIKED_RESIZE = 1280
SP_MAX_KP = 4096
SP_RESIZE = 1600
SP_THRESHOLD = 0.0005

print(f'Device: {device}')


libgomp: Invalid value for environment variable OMP_NUM_THREADS


Device: cuda


## 2. Load Per-Dataset Shortlists

Discover datasets from retrieval output and load shortlist + path mappings.

In [2]:
def load_json(p):
    with open(p) as f: return json.load(f)

# Discover datasets from retrieval output
datasets = {}
for ds_dir in sorted(RETRIEVAL_DIR.iterdir()):
    if ds_dir.is_dir():
        sl_path = ds_dir / 'shortlist.json'
        pm_path = ds_dir / 'image_paths.json'
        if sl_path.exists() and pm_path.exists():
            datasets[ds_dir.name] = {
                'shortlist': load_json(sl_path),
                'path_mapping': load_json(pm_path),
            }

print(f'Datasets with shortlists: {len(datasets)}')
for ds_name, ds_data in datasets.items():
    sl = ds_data['shortlist']
    total = sum(len(v) for v in sl.values())
    print(f'  {ds_name}: {len(sl)} images, {total} pairs')

Datasets with shortlists: 13
  ETs: 22 images, 462 pairs
  amy_gardens: 200 images, 8338 pairs
  fbk_vineyard: 163 images, 7049 pairs
  imc2023_haiper: 54 images, 1804 pairs
  imc2023_heritage: 209 images, 8957 pairs
  imc2023_theather_imc2024_church: 76 images, 2797 pairs
  imc2024_dioscuri_baalshamin: 138 images, 5561 pairs
  imc2024_lizard_pond: 214 images, 8941 pairs
  pt_brandenburg_british_buckingham: 225 images, 9654 pairs
  pt_piazzasanmarco_grandplace: 168 images, 7071 pairs
  pt_sacrecoeur_trevi_tajmahal: 225 images, 9482 pairs
  pt_stpeters_stpauls: 200 images, 8694 pairs
  stairs: 51 images, 1674 pairs


## 3. Load MAST3R Model

In [ ]:
sys.path.insert(0, str(Path('../mast3r')))
from mast3r.model import AsymmetricMASt3R

MAST3R_PATH = '../models/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth'

class FineMASt3R(nn.Module):
    def __init__(self, path):
        super().__init__()
        self.model = AsymmetricMASt3R.from_pretrained(path)
    def forward(self, v1, v2):
        return self.model(v1, v2)

# MASt3R will be loaded lazily in cell-10 (after keypoint extraction, to save GPU)

def empty_cuda_cache():
    if device.type == 'cuda':
        torch.cuda.empty_cache()

def run_mast3r_forward(model, t1, t2):
    """Run MASt3R with CUDA memory hygiene and AMP where available."""
    empty_cuda_cache()
    try:
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
            return model({'img': t1, 'instance': []}, {'img': t2, 'instance': []})
    except torch.cuda.OutOfMemoryError:
        empty_cuda_cache()
        raise

def load_mast3r():
    """Lazy-load MASt3R to GPU. Call after keypoint extraction is done."""
    model = FineMASt3R(MAST3R_PATH).to(device).eval()
    print('MASt3R loaded to GPU')
    return model

match_transform = transforms.Compose([
    transforms.Resize(int(MATCH_SIZE * 1.14)),
    transforms.CenterCrop(MATCH_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])
print('MASt3R helpers ready (model not loaded yet — deferred to cell-10)')

## 3b. Load Keypoint Detectors (ALIKED + SuperPoint)

Used for sparse keypoint extraction in hybrid MASt3R matching.

In [4]:
# Load ALIKED and SuperPoint keypoint detectors
# Champion config: ALIKED (max_kp=4096, resize=1280), SuperPoint (max_kp=4096, threshold=0.0005, resize=1600)

# --- ALIKED ---
aliked_detector = None
try:
    from kornia.feature import ALIKED
    aliked_detector = ALIKED(max_num_keypoints=ALIKED_MAX_KP, detection_threshold=0.01)
    aliked_detector = aliked_detector.to(device).eval()
    print(f'ALIKED loaded via kornia (max_kp={ALIKED_MAX_KP})')
except (ImportError, TypeError) as e:
    print(f'ALIKED kornia failed: {e}')

if aliked_detector is None:
    try:
        from lightglue import ALIKED as LgALIKED
        aliked_detector = LgALIKED(max_num_keypoints=ALIKED_MAX_KP, detection_threshold=0.01, resize=ALIKED_RESIZE)
        aliked_detector = aliked_detector.to(device).eval()
        print(f'ALIKED loaded via lightglue (max_kp={ALIKED_MAX_KP}, resize={ALIKED_RESIZE})')
    except ImportError:
        print('WARNING: ALIKED not available (pip install kornia or lightglue)')

# --- SuperPoint (MagicLeap) ---
superpoint_detector = None
try:
    from kornia.feature import SuperPoint
    superpoint_detector = SuperPoint(pretrained=True).to(device).eval()
    print(f'SuperPoint loaded via kornia')
except ImportError:
    pass

if superpoint_detector is None:
    try:
        from lightglue import SuperPoint as LgSuperPoint
        superpoint_detector = LgSuperPoint(max_num_keypoints=SP_MAX_KP).to(device).eval()
        print(f'SuperPoint loaded via lightglue')
    except ImportError:
        pass

# Fallback: custom SuperPoint implementation
if superpoint_detector is None:
    SP_WEIGHTS_PATH = '../models/superpoint/superpoint_v1.pth'
    if Path(SP_WEIGHTS_PATH).exists():
        class SuperPointNet(nn.Module):
            def __init__(self):
                super().__init__()
                self.encoder = nn.Sequential(
                    nn.Conv2d(1,64,3,padding=1),nn.ReLU(True),nn.Conv2d(64,64,3,padding=1),nn.ReLU(True),nn.MaxPool2d(2,2),
                    nn.Conv2d(64,64,3,padding=1),nn.ReLU(True),nn.Conv2d(64,64,3,padding=1),nn.ReLU(True),nn.MaxPool2d(2,2),
                    nn.Conv2d(64,128,3,padding=1),nn.ReLU(True),nn.Conv2d(128,128,3,padding=1),nn.ReLU(True),nn.MaxPool2d(2,2),
                    nn.Conv2d(128,128,3,padding=1),nn.ReLU(True),nn.Conv2d(128,128,3,padding=1),nn.ReLU(True))
                self.detector = nn.Sequential(nn.Conv2d(128,256,3,padding=1),nn.ReLU(True),nn.Conv2d(256,65,1))
                self.descriptor = nn.Sequential(nn.Conv2d(128,256,3,padding=1),nn.ReLU(True),nn.Conv2d(256,256,1))
            def forward(self, x):
                f = self.encoder(x); return self.detector(f), F.normalize(self.descriptor(f), dim=1)
        
        class SuperPointWrapper(nn.Module):
            def __init__(self, weights_path, max_kp=4096, threshold=0.0005):
                super().__init__()
                self.net = SuperPointNet()
                self.net.load_state_dict(torch.load(weights_path, map_location='cpu', weights_only=False), strict=False)
                self.max_kp = max_kp
                self.threshold = threshold
            @torch.no_grad()
            def forward(self, img):
                gray = img[:, :1, :, :]
                H, W = gray.shape[-2:]
                max_dim = SP_RESIZE
                if max(H, W) > max_dim:
                    scale = max_dim / max(H, W)
                    new_H, new_W = int(H*scale), int(W*scale)
                    gray = F.interpolate(gray, (new_H, new_W), mode='bilinear', align_corners=False)
                score, _ = self.net(gray)
                B, C, Hs, Ws = score.shape
                # SuperPoint has 65 channels (8*8 grid cells + 1 dustbin). Remove dustbin.
                score = score[:, :64, :, :]
                score = score.reshape(B, 8, 8, Hs, Ws).permute(0, 1, 3, 2, 4).reshape(B, Hs*8, Ws*8)
                mask = score > self.threshold
                flat_score = score.reshape(-1)
                flat_mask = mask.reshape(-1)
                valid_scores = flat_score[flat_mask]
                if len(valid_scores) > self.max_kp:
                    thresh = torch.topk(valid_scores, self.max_kp).values[-1]
                    mask = mask & (score >= thresh)
                ys, xs = torch.where(mask[0])
                kpts = torch.stack([xs.float(), ys.float()], dim=-1)
                return {'keypoints': kpts.unsqueeze(0)}
        
        superpoint_detector = SuperPointWrapper(SP_WEIGHTS_PATH, max_kp=SP_MAX_KP, threshold=SP_THRESHOLD).to(device).eval()
        print(f'SuperPoint loaded via custom impl (max_kp={SP_MAX_KP}, threshold={SP_THRESHOLD}, resize={SP_RESIZE})')

if superpoint_detector is None:
    print('WARNING: SuperPoint not available (pip install kornia or lightglue)')

print('Keypoint detectors ready')

ALIKED loaded via kornia (max_kp=4096)
SuperPoint loaded via custom impl (max_kp=4096, threshold=0.0005, resize=1600)
Keypoint detectors ready


## 4. Matching Functions (Dense + Sparse Hybrid)

**Dense**: MASt3R semi-dense matching via mutual nearest neighbors (subsample=8, pixel_tol=5).
**Sparse**: Detect keypoints (ALIKED/SuperPoint) → interpolate MASt3R descriptors at kp positions → reciprocal NN matching.

In [5]:
@torch.no_grad()
def match_pair(model, p1, p2, transform):
    """MAST3R dense matching via mutual nearest neighbors."""
    img1 = Image.open(p1).convert('RGB')
    img2 = Image.open(p2).convert('RGB')
    H1, W1 = img1.height, img1.width
    H2, W2 = img2.height, img2.width

    t1 = transform(img1).unsqueeze(0).to(device)
    t2 = transform(img2).unsqueeze(0).to(device)

    result = {'matches': [], 'mkpts1': [], 'mkpts2': [], 'num_matches': 0}
    try:
        res1, res2 = run_mast3r_forward(model, t1, t2)
    except torch.cuda.OutOfMemoryError:
        print(f'    dense OOM skipped: {Path(p1).name} :: {Path(p2).name}')
        return result

    def _unwrap(desc_raw):
        """MAST3R returns (B, H, W, D) → convert to (D, H, W); B may be 2"""
        if desc_raw is None:
            return None
        t = desc_raw[-1] if isinstance(desc_raw, (list, tuple)) else desc_raw
        t = t[0]  # take first batch element (MAST3R may return batch=2)
        if t.dim() == 3:
            if t.shape[-1] < t.shape[0] and t.shape[-1] < t.shape[1]:
                t = t.permute(2, 0, 1)  # (H,W,D) → (D,H,W)
            return t
        if t.dim() == 2:
            N, D = t.shape
            s = int(N ** 0.5)
            if s * s == N:
                return t.reshape(s, s, D).permute(2, 0, 1)
            return None
        return None

    def _unwrap_conf(c):
        if c is None: return None
        return c[0]  # (B, H, W) → (H, W)

    d1 = _unwrap(res1.get('desc'))
    d2 = _unwrap(res2.get('desc'))
    c1 = _unwrap_conf(res1.get('conf'))
    c2 = _unwrap_conf(res2.get('conf'))

    if d1 is None or d2 is None:
        return result

    D1, Hd1, Wd1 = d1.shape
    D2, Hd2, Wd2 = d2.shape

    # Subsample
    ys1 = torch.arange(0, Hd1, SUBSAMPLE, device=device)
    xs1 = torch.arange(0, Wd1, SUBSAMPLE, device=device)
    ys2 = torch.arange(0, Hd2, SUBSAMPLE, device=device)
    xs2 = torch.arange(0, Wd2, SUBSAMPLE, device=device)

    d1_s = d1[:, ys1[:, None], xs1[None, :]]
    d2_s = d2[:, ys2[:, None], xs2[None, :]]
    d1_flat = d1_s.reshape(D1, -1).t()
    d2_flat = d2_s.reshape(D2, -1).t()
    d1_flat = F.normalize(d1_flat, dim=-1)
    d2_flat = F.normalize(d2_flat, dim=-1)

    # Mutual nearest neighbors
    sim = torch.mm(d1_flat, d2_flat.t())
    nn12 = sim.argmax(dim=1)
    nn21 = sim.argmax(dim=0)
    mutual = torch.zeros(len(nn12), dtype=torch.bool, device=device)
    for i in range(len(nn12)):
        if nn21[nn12[i]] == i:
            mutual[i] = True
    if mutual.sum() == 0:
        return result

    idx1 = torch.where(mutual)[0]
    idx2 = nn12[mutual]
    h_scale1, w_scale1 = H1 / Hd1, W1 / Wd1
    h_scale2, w_scale2 = H2 / Hd2, W2 / Wd2
    y1 = (idx1 // (Wd1 // SUBSAMPLE)) * SUBSAMPLE * h_scale1
    x1 = (idx1 % (Wd1 // SUBSAMPLE)) * SUBSAMPLE * w_scale1
    y2 = (idx2 // (Wd2 // SUBSAMPLE)) * SUBSAMPLE * h_scale2
    x2 = (idx2 % (Wd2 // SUBSAMPLE)) * SUBSAMPLE * w_scale2

    mkpts1 = torch.stack([x1, y1], dim=-1).cpu().numpy()
    mkpts2 = torch.stack([x2, y2], dim=-1).cpu().numpy()
    matches = np.concatenate([mkpts1, mkpts2], axis=1)
    result['matches'] = [list(map(float, m)) for m in matches]
    result['mkpts1'] = [list(map(float, m)) for m in mkpts1]
    result['mkpts2'] = [list(map(float, m)) for m in mkpts2]
    result['num_matches'] = len(matches)
    return result

print('Matching function ready')

Matching function ready


In [6]:
@torch.no_grad()
def match_sparse_via_mast3r(model, detector, p1, p2, transform, detector_name='kp',
                             max_kp=4096, resize_to=None):
    """Extract keypoints via detector, then match using MASt3R descriptor interpolation."""
    img1 = Image.open(p1).convert('RGB')
    img2 = Image.open(p2).convert('RGB')
    H1, W1 = img1.height, img1.width
    H2, W2 = img2.height, img2.width
    
    t1 = transform(img1).unsqueeze(0).to(device)
    t2 = transform(img2).unsqueeze(0).to(device)
    
    result = {'matches': [], 'num_matches': 0}
    try:
        res1, res2 = run_mast3r_forward(model, t1, t2)
    except torch.cuda.OutOfMemoryError:
        print(f'    sparse OOM skipped: {Path(p1).name} :: {Path(p2).name}')
        return result
    
    def _unwrap(desc_raw):
        if desc_raw is None: return None
        t = desc_raw[-1] if isinstance(desc_raw, (list, tuple)) else desc_raw
        t = t[0]
        if t.dim() == 3:
            if t.shape[-1] < t.shape[0] and t.shape[-1] < t.shape[1]:
                t = t.permute(2, 0, 1)
            return t
        return None
    
    d1 = _unwrap(res1.get('desc'))
    d2 = _unwrap(res2.get('desc'))
    
    if d1 is None or d2 is None:
        return result
    
    D_dim, Hd1, Wd1 = d1.shape
    _, Hd2, Wd2 = d2.shape
    
    def detect_kps(detector, img_pil, max_kp, resize_to_val):
        """Detect keypoints, return (N, 2) tensor in original image coordinates."""
        img_tensor = transforms.ToTensor()(img_pil).unsqueeze(0).to(device)
        H_orig, W_orig = img_tensor.shape[-2:]
        
        if resize_to_val is not None and max(H_orig, W_orig) > resize_to_val:
            scale = resize_to_val / max(H_orig, W_orig)
            new_H, new_W = int(H_orig * scale), int(W_orig * scale)
            img_resized = F.interpolate(img_tensor, (new_H, new_W), mode='bilinear', align_corners=False)
        else:
            img_resized = img_tensor
            scale = 1.0
        
        try:
            output = detector(img_resized)
            # Handle different return types from different detector implementations
            if hasattr(output, 'keypoints'):
                # kornia returns dataclass: ALIKEDFeatures, SuperPointFeatures
                kpts = output.keypoints
            elif isinstance(output, dict):
                kpts = output['keypoints']
            elif isinstance(output, (list, tuple)):
                kpts = output[0]
            else:
                kpts = output
            
            if hasattr(kpts, 'reshape'):
                kpts = kpts.reshape(-1, 2) if kpts.dim() == 3 else kpts
                if kpts.dim() == 3:
                    kpts = kpts[0]
            elif hasattr(kpts, 'size'):
                kpts = torch.as_tensor(kpts)
        except Exception:
            return torch.zeros((0, 2), device=device)
        
        if not hasattr(kpts, 'numel') or kpts.numel() == 0:
            return kpts if hasattr(kpts, 'numel') else torch.zeros((0, 2), device=device)
        
        kpts = kpts / scale
        if len(kpts) > max_kp:
            idxs = torch.randperm(len(kpts), device=device)[:max_kp]
            kpts = kpts[idxs]
        return kpts
    
    kp1 = detect_kps(detector, img1, max_kp, resize_to)
    kp2 = detect_kps(detector, img2, max_kp, resize_to)
    
    if len(kp1) == 0 or len(kp2) == 0:
        return result
    
    # Scale keypoints to descriptor map then bilinear interpolation
    kp1_desc = kp1.clone()
    kp1_desc[:, 0] = kp1[:, 0] * (Wd1 / W1)
    kp1_desc[:, 1] = kp1[:, 1] * (Hd1 / H1)
    kp2_desc = kp2.clone()
    kp2_desc[:, 0] = kp2[:, 0] * (Wd2 / W2)
    kp2_desc[:, 1] = kp2[:, 1] * (Hd2 / H2)
    
    kp1_norm = kp1_desc.clone()
    kp1_norm[:, 0] = kp1_norm[:, 0] / (Wd1 - 1) * 2 - 1
    kp1_norm[:, 1] = kp1_norm[:, 1] / (Hd1 - 1) * 2 - 1
    kp2_norm = kp2_desc.clone()
    kp2_norm[:, 0] = kp2_norm[:, 0] / (Wd2 - 1) * 2 - 1
    kp2_norm[:, 1] = kp2_norm[:, 1] / (Hd2 - 1) * 2 - 1
    
    grid1 = kp1_norm.unsqueeze(0).unsqueeze(0)
    grid2 = kp2_norm.unsqueeze(0).unsqueeze(0)
    
    desc1_at_kp = F.grid_sample(d1.unsqueeze(0), grid1, mode='bilinear', align_corners=True)
    desc1_at_kp = desc1_at_kp[0, :, 0, :].t()
    desc2_at_kp = F.grid_sample(d2.unsqueeze(0), grid2, mode='bilinear', align_corners=True)
    desc2_at_kp = desc2_at_kp[0, :, 0, :].t()
    
    desc1_at_kp = F.normalize(desc1_at_kp, dim=-1)
    desc2_at_kp = F.normalize(desc2_at_kp, dim=-1)
    
    sim = torch.mm(desc1_at_kp, desc2_at_kp.t())
    nn12 = sim.argmax(dim=1)
    nn21 = sim.argmax(dim=0)
    
    mutual = torch.zeros(len(nn12), dtype=torch.bool, device=device)
    for i in range(len(nn12)):
        if nn21[nn12[i]] == i:
            mutual[i] = True
    if mutual.sum() == 0:
        return result
    
    idx1 = torch.where(mutual)[0]
    idx2 = nn12[mutual]
    scores = sim[idx1, idx2]
    valid = scores > MATCH_THRESHOLD
    idx1 = idx1[valid]; idx2 = idx2[valid]
    
    if len(idx1) == 0:
        return result
    
    x1, y1 = kp1[idx1, 0], kp1[idx1, 1]
    x2, y2 = kp2[idx2, 0], kp2[idx2, 1]
    matches = torch.stack([x1, y1, x2, y2], dim=-1).cpu().numpy()
    result['matches'] = [list(map(float, m)) for m in matches]
    result['num_matches'] = len(matches)
    return result

print('Sparse matching function ready')

Sparse matching function ready


In [7]:
@torch.no_grad()
def match_pair_hybrid(model, p1, p2, transform, aliked_det=None, sp_det=None):
    """Hybrid MASt3R matching: dense semi-dense + ALIKED sparse + SuperPoint sparse.
    
    Returns combined matches from all three sources (union, with deduplication).
    """
    all_matches_list = []
    
    # 1. Dense MASt3R matching
    dense_result = match_pair(model, p1, p2, transform)
    if dense_result['num_matches'] > 0:
        all_matches_list.append(dense_result['matches'])
    
    # 2. ALIKED sparse matching (keypoints matched via MASt3R descriptors)
    if aliked_det is not None:
        aliked_result = match_sparse_via_mast3r(
            model, aliked_det, p1, p2, transform,
            detector_name='aliked', max_kp=ALIKED_MAX_KP, resize_to=ALIKED_RESIZE)
        if aliked_result['num_matches'] > 0:
            all_matches_list.append(aliked_result['matches'])
    
    # 3. SuperPoint sparse matching (keypoints matched via MASt3R descriptors)
    if sp_det is not None:
        sp_result = match_sparse_via_mast3r(
            model, sp_det, p1, p2, transform,
            detector_name='superpoint', max_kp=SP_MAX_KP, resize_to=SP_RESIZE)
        if sp_result['num_matches'] > 0:
            all_matches_list.append(sp_result['matches'])
    
    # Combine all matches (deduplicate by rounding to integer pixel coords)
    if not all_matches_list:
        return {'matches': [], 'num_matches': 0}
    
    combined = []
    seen = set()
    for match_list in all_matches_list:
        for m in match_list:
            # Dedup key: rounded integer coordinates
            key = (round(m[0]), round(m[1]), round(m[2]), round(m[3]))
            if key not in seen:
                seen.add(key)
                combined.append(m)
    
    return {'matches': combined, 'num_matches': len(combined)}

print('Hybrid matching function ready')

Hybrid matching function ready


In [8]:
@torch.no_grad()
def match_sparse_using_keypoints(model, kp1, kp2, p1, p2, transform, max_kp=4096):
    """Match pre-extracted keypoints with MASt3R descriptor interpolation.

    Returns both coordinate matches (mkpts1/mkpts2) and matched_idx into the
    original per-image keypoint arrays, so COLMAP can build stable tracks.
    """
    img1 = Image.open(p1).convert('RGB')
    img2 = Image.open(p2).convert('RGB')
    H1, W1 = img1.height, img1.width
    H2, W2 = img2.height, img2.width

    t1 = transform(img1).unsqueeze(0).to(device)
    t2 = transform(img2).unsqueeze(0).to(device)
    result = {'matches': [], 'mkpts1': [], 'mkpts2': [], 'matched_idx': [], 'scores': [], 'num_matches': 0}
    try:
        res1, res2 = run_mast3r_forward(model, t1, t2)
    except torch.cuda.OutOfMemoryError:
        print(f'    sparse OOM skipped: {Path(p1).name} :: {Path(p2).name}')
        return result

    def _unwrap(desc_raw):
        if desc_raw is None:
            return None
        t = desc_raw[-1] if isinstance(desc_raw, (list, tuple)) else desc_raw
        t = t[0]
        if t.dim() == 3:
            if t.shape[-1] < t.shape[0] and t.shape[-1] < t.shape[1]:
                t = t.permute(2, 0, 1)
            return t
        return None

    d1 = _unwrap(res1.get('desc'))
    d2 = _unwrap(res2.get('desc'))
    if d1 is None or d2 is None:
        return result

    _, Hd1, Wd1 = d1.shape
    _, Hd2, Wd2 = d2.shape

    kp1 = kp1.to(device).float()
    kp2 = kp2.to(device).float()
    sel1 = torch.arange(len(kp1), device=device)
    sel2 = torch.arange(len(kp2), device=device)
    if len(kp1) > max_kp:
        sel1 = torch.randperm(len(kp1), device=device)[:max_kp]
        kp1 = kp1[sel1]
    if len(kp2) > max_kp:
        sel2 = torch.randperm(len(kp2), device=device)[:max_kp]
        kp2 = kp2[sel2]
    if len(kp1) == 0 or len(kp2) == 0:
        return result

    kp1_desc = kp1.clone()
    kp1_desc[:, 0] = kp1[:, 0] * (Wd1 / W1)
    kp1_desc[:, 1] = kp1[:, 1] * (Hd1 / H1)
    kp2_desc = kp2.clone()
    kp2_desc[:, 0] = kp2[:, 0] * (Wd2 / W2)
    kp2_desc[:, 1] = kp2[:, 1] * (Hd2 / H2)

    kp1_norm = kp1_desc.clone()
    kp1_norm[:, 0] = kp1_norm[:, 0] / (Wd1 - 1) * 2 - 1
    kp1_norm[:, 1] = kp1_norm[:, 1] / (Hd1 - 1) * 2 - 1
    kp2_norm = kp2_desc.clone()
    kp2_norm[:, 0] = kp2_norm[:, 0] / (Wd2 - 1) * 2 - 1
    kp2_norm[:, 1] = kp2_norm[:, 1] / (Hd2 - 1) * 2 - 1

    desc1 = F.grid_sample(d1.unsqueeze(0), kp1_norm.unsqueeze(0).unsqueeze(0), mode='bilinear', align_corners=True)
    desc2 = F.grid_sample(d2.unsqueeze(0), kp2_norm.unsqueeze(0).unsqueeze(0), mode='bilinear', align_corners=True)
    desc1 = F.normalize(desc1[0, :, 0, :].t(), dim=-1)
    desc2 = F.normalize(desc2[0, :, 0, :].t(), dim=-1)

    sim = torch.mm(desc1, desc2.t())
    nn12 = sim.argmax(dim=1)
    nn21 = sim.argmax(dim=0)
    mutual = torch.zeros(len(nn12), dtype=torch.bool, device=device)
    for i in range(len(nn12)):
        if nn21[nn12[i]] == i:
            mutual[i] = True
    if mutual.sum() == 0:
        return result

    idx1 = torch.where(mutual)[0]
    idx2 = nn12[mutual]
    scores = sim[idx1, idx2]
    valid = scores > MATCH_THRESHOLD
    idx1 = idx1[valid]
    idx2 = idx2[valid]
    scores = scores[valid]
    if len(idx1) == 0:
        return result

    mkpts1 = kp1[idx1]
    mkpts2 = kp2[idx2]
    matched_idx = torch.stack([sel1[idx1], sel2[idx2]], dim=-1)
    matches = torch.cat([mkpts1, mkpts2], dim=-1).cpu().numpy()

    result['matches'] = [list(map(float, m)) for m in matches]
    result['mkpts1'] = [list(map(float, m)) for m in mkpts1.cpu().numpy()]
    result['mkpts2'] = [list(map(float, m)) for m in mkpts2.cpu().numpy()]
    result['matched_idx'] = [list(map(int, m)) for m in matched_idx.cpu().numpy()]
    result['scores'] = [float(s) for s in scores.detach().cpu().numpy()]
    result['num_matches'] = len(matches)
    return result

print('Per-image keypoint matching function ready')

Per-image keypoint matching function ready


## 5a. Per-Image Keypoint Extraction (DataLoader, batched)

Run ALIKED + SuperPoint on all images before MASt3R is loaded (GPU free).

In [ ]:
# ================================================================
# Step 1: Extract keypoints ONCE per image (shared across all pairs)
# DataLoader batched processing — pad to batch max size, resize to detector max_dim
# MASt3R not loaded yet, GPU fully available
# ================================================================
print('Extracting per-image keypoints with DataLoader (batched, MASt3R not loaded)...')
from torch.utils.data import Dataset, DataLoader

class ImageDataset(Dataset):
    def __init__(self, img_paths):
        self.img_paths = img_paths
    def __len__(self):
        return len(self.img_paths)
    def __getitem__(self, idx):
        p = self.img_paths[idx]
        img = transforms.ToTensor()(Image.open(p).convert('RGB'))
        return img, str(p)

def collate_to_max_size(batch):
    """Pad images to same H×W, track per-image original size before padding."""
    imgs, paths = zip(*batch)
    max_h = max(img.shape[1] for img in imgs)
    max_w = max(img.shape[2] for img in imgs)
    orig_sizes = [(img.shape[1], img.shape[2]) for img in imgs]
    padded = []
    for img in imgs:
        ph, pw = max_h - img.shape[1], max_w - img.shape[2]
        padded.append(F.pad(img, (0, pw, 0, ph)) if ph > 0 or pw > 0 else img)
    return torch.stack(padded), list(paths), orig_sizes

@torch.no_grad()
def extract_kps_batch(img_batch, detector, max_dim, orig_sizes):
    """Detect keypoints from a padded batch. orig_sizes is list of (H, W) per image."""
    kps_list = [torch.zeros((0, 2)) for _ in range(img_batch.shape[0])]
    try:
        H, W = img_batch.shape[-2:]
        if max(H, W) > max_dim:
            scale = max_dim / max(H, W)
            img_r = F.interpolate(img_batch, (int(H * scale), int(W * scale)), mode='bilinear', align_corners=False)
        else:
            img_r, scale = img_batch, 1.0
        out = detector(img_r)
        if isinstance(out, (list, tuple)):
            out = out[0]
        kps = out.keypoints if hasattr(out, 'keypoints') else out['keypoints'] if isinstance(out, dict) else out
        if isinstance(kps, torch.Tensor):
            if kps.dim() == 2:
                kps = kps.unsqueeze(0)
            for i in range(min(len(kps_list), kps.shape[0])):
                ki = kps[i] / scale
                oh, ow = orig_sizes[i]
                mask = (ki[:, 0] < ow) & (ki[:, 1] < oh)
                kps_list[i] = ki[mask].cpu()
    except Exception as e:
        print(f'    {type(detector).__name__} batch failed: {type(e).__name__}: {e}')
    return kps_list

aliked_keypoints = {}
sp_keypoints = {}
BATCH_SIZE = 8

for ds_name, ds_data in tqdm(datasets.items(), desc='Keypoints'):
    path_mapping = ds_data['path_mapping']
    all_images = sorted(path_mapping.keys())
    all_paths = [str(Path(path_mapping[n])) for n in all_images if path_mapping.get(n) and Path(path_mapping[n]).exists()]

    if not all_paths:
        aliked_keypoints[ds_name] = {}
        sp_keypoints[ds_name] = {}
        continue

    dataset = ImageDataset(all_paths)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_to_max_size, num_workers=0)

    aliked_kps = {}
    sp_kps = {}

    for img_batch, path_batch, orig_sizes in tqdm(loader, desc=f'  {ds_name}', leave=False):
        img_batch = img_batch.to(device)

        if aliked_detector is not None:
            for p, kps in zip(path_batch, extract_kps_batch(img_batch, aliked_detector, ALIKED_RESIZE, orig_sizes)):
                if kps.numel() > 0:
                    aliked_kps[Path(p).name] = kps

        if superpoint_detector is not None:
            for p, kps in zip(path_batch, extract_kps_batch(img_batch, superpoint_detector, SP_RESIZE, orig_sizes)):
                if kps.numel() > 0:
                    sp_kps[Path(p).name] = kps

        del img_batch
        torch.cuda.empty_cache()

    aliked_keypoints[ds_name] = aliked_kps
    sp_keypoints[ds_name] = sp_kps
    aliked_avg = sum(len(v) for v in aliked_kps.values()) // max(1, len(aliked_kps))
    sp_avg = sum(len(v) for v in sp_kps.values()) // max(1, len(sp_kps))
    print(f'  {ds_name}: ALIKED avg {aliked_avg} kp/img, SP avg {sp_avg} kp/img')

print('Keypoint extraction complete.')

## 5b. MASt3R Hybrid Matching

Release detectors → load MASt3R → match all pairs for each dataset.

In [ ]:
# ================================================================
# Step 2: Release detectors, load MASt3R, run hybrid matching
# ================================================================
print(f'Hybrid MASt3R matching per dataset...')
print(f'  Dense: subsample={SUBSAMPLE}, pixel_tol={PIXEL_TOL}')
print(f'  ALIKED: max_kp={ALIKED_MAX_KP}, resize={ALIKED_RESIZE}')
print(f'  SuperPoint: max_kp={SP_MAX_KP}, resize={SP_RESIZE}, threshold={SP_THRESHOLD}')
print(f'  Match threshold: {MATCH_THRESHOLD}')

# Keypoints already extracted — delete detectors to free GPU for MASt3R
if aliked_detector is not None:
    del aliked_detector
    aliked_detector = None
if superpoint_detector is not None:
    del superpoint_detector
    superpoint_detector = None
gc.collect()
torch.cuda.empty_cache()
print(f'  Detectors released, GPU ready for MASt3R')

mast3r = load_mast3r()

def tensor_to_list(x):
    if isinstance(x, torch.Tensor):
        return [[float(v) for v in row] for row in x.detach().cpu().numpy().reshape(-1, 2)]
    return []

t0_total = time.time()

for ds_name, ds_data in tqdm(datasets.items(), desc='Datasets'):
    ds_out = OUTPUT_DIR / ds_name
    ds_out.mkdir(parents=True, exist_ok=True)

    shortlist = ds_data['shortlist']
    path_mapping = ds_data['path_mapping']
    pairs = set()
    for img, cands in shortlist.items():
        for c in cands:
            pairs.add(tuple(sorted([img, c['name']])))
    pairs = list(pairs)

    keypoint_payload = {
        'aliked': {name: tensor_to_list(kps) for name, kps in aliked_keypoints.get(ds_name, {}).items()},
        'superpoint': {name: tensor_to_list(kps) for name, kps in sp_keypoints.get(ds_name, {}).items()},
    }
    with open(ds_out / 'keypoints.json', 'w') as f:
        json.dump(keypoint_payload, f)

    print(f'\n{ds_name}: {len(shortlist)} images, {len(pairs)} pairs')
    t0 = time.time()
    hybrid_matches = {}
    dense_matches_compat = {}

    for n1, n2 in tqdm(pairs, desc=f'  {ds_name}', leave=False):
        p1 = path_mapping.get(n1)
        p2 = path_mapping.get(n2)
        if not p1 or not p2:
            continue

        dense_result = match_pair(mast3r, p1, p2, match_transform)
        sparse_results = {}
        combined_lists = []
        if dense_result['num_matches'] > 0:
            combined_lists.append(dense_result['matches'])

        ak = aliked_keypoints.get(ds_name, {})
        if ak and n1 in ak and n2 in ak:
            aliked_result = match_sparse_using_keypoints(mast3r, ak[n1], ak[n2], p1, p2, match_transform, max_kp=ALIKED_MAX_KP)
            sparse_results['aliked'] = aliked_result
            if aliked_result['num_matches'] > 0:
                combined_lists.append(aliked_result['matches'])

        sk = sp_keypoints.get(ds_name, {})
        if sk and n1 in sk and n2 in sk:
            sp_result = match_sparse_using_keypoints(mast3r, sk[n1], sk[n2], p1, p2, match_transform, max_kp=SP_MAX_KP)
            sparse_results['superpoint'] = sp_result
            if sp_result['num_matches'] > 0:
                combined_lists.append(sp_result['matches'])

        combined = []
        seen = set()
        for match_list in combined_lists:
            for m in match_list:
                key = (round(m[0]), round(m[1]), round(m[2]), round(m[3]))
                if key not in seen:
                    seen.add(key)
                    combined.append(m)

        if len(combined) >= MIN_MATCHES:
            pair_key = f'{n1}::{n2}'
            hybrid_matches[pair_key] = {
                'image1': n1,
                'image2': n2,
                'dense': dense_result,
                'sparse': sparse_results,
                'matches': combined,
                'num_matches': len(combined),
            }
            dense_matches_compat[pair_key] = {'matches': combined, 'num_matches': len(combined)}

        if EMPTY_CACHE_EVERY_PAIR:
            empty_cuda_cache()

    with open(ds_out / 'hybrid_matches.json', 'w') as f:
        json.dump(hybrid_matches, f)
    with open(ds_out / 'dense_matches.json', 'w') as f:
        json.dump(dense_matches_compat, f)

    total_m = sum(r['num_matches'] for r in hybrid_matches.values())
    total_sparse = sum(
        r.get('num_matches', 0)
        for pair in hybrid_matches.values()
        for r in pair.get('sparse', {}).values()
    )
    print(f'  {ds_name}: {len(hybrid_matches)}/{len(pairs)} pairs matched, '
          f'{total_m} total matches ({total_sparse} sparse), done in {time.time()-t0:.0f}s')

print(f'\nAll matching done in {time.time()-t0_total:.0f}s')

## 6. Summary

| Parameter | Value |
|-----------|-------|
| MASt3R Image size | 512 |
| Dense subsample | 8 |
| Dense pixel tolerance | 5 |
| Match threshold | 1.001 |
| Min matches per pair | 15 |
| ALIKED max keypoints | 4096 |
| ALIKED resize | 1280 |
| SuperPoint max keypoints | 4096 |
| SuperPoint resize | 1600 |
| SuperPoint threshold | 0.0005 |
| **Matching type** | **Hybrid (Dense + ALIKED + SuperPoint)** |

Next → `03_colmap_reconstruction.ipynb`